In [5]:
import json
import numpy as np
import pandas as pd
from io import StringIO
import textwrap
from model_inference.gpt import *
from utils.table_utils import *

# Table parsing test

In [19]:
path = 'data/livesum/test.json'
df = pd.read_json(path)
idx = 4

In [20]:
table_string = df['table'][idx]
table_string = table_string.replace('<NEWLINE>', '\n')
table_string_io = StringIO(table_string)
df_table = pd.read_csv(table_string_io)

In [22]:
print(df_table.to_string(index=False))

     Team  Goals  Shots  Fouls  Yellow Cards  Red Cards  Corner Kicks  Free Kicks  Offsides
Away Team      3     16     13             2          0             3          18         0
Home Team      0     28     18             2          0             7          12         2


In [9]:
import re

def paragraph_to_numbered_sentences(paragraph):
    # Split paragraph into sentences using regular expression to handle various punctuation
    sentences = re.split(r'(?<!\w\.\w.)(?<![A-Z][a-z]\.)(?<=\.|\?|\!)\s', paragraph)
    
    # Number each sentence and store it in a list
    numbered_sentences = [f"{i+1}. {sentence.strip()}" for i, sentence in enumerate(sentences) if sentence.strip()]
    return numbered_sentences

# Example paragraph
paragraph = "The game kicks off with the start of the first half. Player11's shot is saved. The Away Team takes the lead with a goal!"

formatted_output = "\n".join(paragraph_to_numbered_sentences(paragraph= df['text'][idx]))


In [10]:
print(formatted_output)

1. And we're off for the first half.
2. Player27(Away Team)'s shot from close range is blocked.
3. Player27(Away Team)'s shot from the center of the box is blocked with the assistance of Player26(Away Team).
4. The Away Team earns a corner kick.
5. Offside called against the Away Team as Player24(Away Team) attempts a pass to Player32(Away Team), but Player28(Away Team) is caught in an offside position.
6. The Away Team wins a corner kick.
7. Player29(Away Team) narrowly misses with a right-footed shot from outside the box, assisted by Player23(Away Team) after a corner, as Player6(Home Team) is down with an injury causing a delay in the match.
8. The delay is finished and they are prepared to resume play.
9. Player28(Away Team) earns a free kick after being fouled by Player8(Home Team) on the left wing.
10. Player20(Away Team) is causing a delay in the match due to an injury.
11. The delay has ended and they are prepared to resume play.
12. Player26(Away Team)'s shot from outside the 

# Prompting test

In [11]:
df = pd.read_json('data/livesum/test.json')
%clear
print(textwrap.fill(df['text'][idx], width=100))

And we're off for the first half. Player27(Away Team)'s shot from close range is blocked.
Player27(Away Team)'s shot from the center of the box is blocked with the assistance of
Player26(Away Team). The Away Team earns a corner kick. Offside called against the Away Team as
Player24(Away Team) attempts a pass to Player32(Away Team), but Player28(Away Team) is caught in an
offside position. The Away Team wins a corner kick. Player29(Away Team) narrowly misses with a
right-footed shot from outside the box, assisted by Player23(Away Team) after a corner, as
Player6(Home Team) is down with an injury causing a delay in the match. The delay is finished and
they are prepared to resume play. Player28(Away Team) earns a free kick after being fouled by
Player8(Home Team) on the left wing. Player20(Away Team) is causing a delay in the match due to an
injury. The delay has ended and they are prepared to resume play. Player26(Away Team)'s shot from
outside the box is saved in the middle of the goal.

In [12]:
text = df['text'][idx]
atomic_out = ask_chatgpt(text=text,prompt_path="prompts/Livesum/livesum_atomic.txt")
print(atomic_out)

The first half of the match begins.  
Player27 from the Away Team takes a shot from close range.  
The shot by Player27 from the Away Team is blocked.  
Player27 from the Away Team takes a shot from the center of the box.  
The shot by Player27 from the Away Team is blocked with the assistance of Player26 from the Away Team.  
The Away Team earns a corner kick.  
Player24 from the Away Team attempts a pass to Player32 from the Away Team.  
Player28 from the Away Team is caught in an offside position.  
Offside is called against the Away Team.  
The Away Team wins a corner kick.  
Player29 from the Away Team takes a right-footed shot from outside the box.  
The shot by Player29 from the Away Team narrowly misses.  
Player23 from the Away Team assists Player29 from the Away Team after a corner.  
Player6 from the Home Team is down with an injury.  
The injury causes a delay in the match.  
The delay in the match is finished.  
The teams are prepared to resume play.  
Player28 from the Aw

In [13]:
with open('./model_outputs/gpt_livesum_test/atomic_each.txt', 'w') as f:
    f.write(atomic_out)

### Schema Generation

In [14]:
for idx in range(0,5):
    #atomic_text = "\n".join(paragraph_to_numbered_sentences(paragraph= df['text'][idx]))
    atomic_out = ask_chatgpt(text=df['text'][idx],prompt_path="prompts/Livesum/livesum_atomic.txt")
    header_out = ask_chatgpt(text=atomic_out,prompt_path="prompts/Livesum/livesum_header.txt")
    with open(f'./model_outputs/Livesum/GPT4o_Headers/{idx}.txt', 'w') as f:
        f.write(header_out)
    print(header_out)

```json
{
  "row_headers": ["Home Team", "Away Team"],
  "column_headers": [
    "Team",
    "Goals",
    "Shots",
    "Shots on Target",
    "Shots off Target",
    "Shots Blocked",
    "Assists",
    "Fouls Committed",
    "Fouls Suffered",
    "Yellow Cards",
    "Offsides",
    "Free Kicks Won",
    "Corner Kicks"
  ]
}
```
```json
{
  "row_headers": ["Home Team", "Away Team"],
  "column_headers": [
    "Team",
    "Goals",
    "Fouls",
    "Free Kicks",
    "Offsides",
    "Shots",
    "Shots on Target",
    "Shots Blocked",
    "Shots Missed",
    "Yellow Cards",
    "Corner Kicks"
  ]
}
```
```json
{
  "row_headers": ["Home Team", "Away Team"],
  "column_headers": [
    "Team",
    "Goals",
    "Fouls",
    "Yellow Cards",
    "Red Cards",
    "Offsides",
    "Corner Kicks",
    "Shots",
    "Shots on Target",
    "Shots Blocked",
    "Free Kicks"
  ]
}
```
```json
{
  "row_headers": ["Home Team", "Away Team"],
  "column_headers": [
    "Team",
    "Goals",
    "Shots",
    "Sho

### Baselines

In [15]:
for idx in range(0,5):
    formatted_output = "\n".join(paragraph_to_numbered_sentences(paragraph= df['text'][idx]))
    input_text = formatted_output 
    output_table = ask_chatgpt(text=input_text,prompt_path="prompts/Livesum/baseline.txt")
    with open(f"model_outputs/Livesum/GPT4o-Baseline/{idx}.txt",'w+') as f:
        f.write(output_table)
    #print(output_table)
    print('Saved results for idx',idx)
    print("*****************************")
        
#print(output_table)

FileNotFoundError: [Errno 2] No such file or directory: 'model_outputs/Livesum/GPT4o-Baseline/0.txt'

### Table Filling (our method)

In [ ]:
# with open('./model_outputs/gpt_livesum_test/header_each.txt', 'r') as f:
#     header_text = f.read()
# with open('./model_outputs/gpt_livesum_test/atomic_each.txt', 'r') as f:
#     atomic_text = f.read()
for idx in range(0,5):
    formatted_output = "\n".join(paragraph_to_numbered_sentences(paragraph= df['text'][idx]))
    input_text = header_out + '\n' + formatted_output 
    output_table = ask_chatgpt(text=input_text,prompt_path="prompts/Livesum/fill_table.txt")
    with open(f"model_outputs/Livesum_Gpt4o/{idx}.txt",'w+') as f:
        f.write(output_table)
    print('Saved results for idx',idx)
    print("*****************************")
        
#print(output_table)

### Table Filling (2 step- Our method)

In [4]:
import pandas as pd
import re

import pandas as pd
import re

def extract_half_table(file_contents: str) -> pd.DataFrame:
    # Locate the start of the final table section
    start_match = re.search(r"###\s*Final\s*Table", file_contents)
    if not start_match:
        raise ValueError("No '### Final Table' heading found.")
        
    # Extract the substring starting from ### Final Table
    final_table_start = start_match.end()
    text_after_final_table = file_contents[final_table_start:].strip()
    
    # Split lines and filter lines containing '<NEWLINE>'
    lines = [line.strip() for line in text_after_final_table.splitlines() if '<NEWLINE>' in line]
    
    if not lines:
        raise ValueError("No table lines found in the final table section.")
    
    # Extract header and clean up columns
    header_line = lines[0]
    columns = [col.strip() for col in header_line.split('|') if col.strip() and col.strip() != '<NEWLINE>']
    
    # Process data rows
    data = []
    for line in lines[1:]:
        parts = [part.strip() for part in line.split('|') if part.strip() and part.strip() != '<NEWLINE>']
        # Debug: Print row length for comparison with header
        if len(parts) != len(columns):
            print(f"Row length mismatch! Header columns: {len(columns)}, Row data: {len(parts)}")
            print(f"Row content: {parts}")
        # Replace 'Not found' with 0 for numeric columns
        processed_parts = [0 if part == 'Not found' else part for part in parts]
        data.append(processed_parts)
    
    # Separate team names from the rest of the data
    team_names = [row[0] for row in data]
    data_values = [row[1:] for row in data]
    
    # Check consistency of all rows with the header
    for idx, row in enumerate(data_values):
        if len(row) != len(columns) - 1:  # Subtract 1 for the team name
            raise ValueError(f"Row {idx + 1} has {len(row)} values, but {len(columns) - 1} were expected. Row content: {row}")
    
    # Create DataFrame
    df = pd.DataFrame(data_values, index=team_names, columns=columns[1:])
    
    # Replace 'Not found' and convert numeric columns
    df.replace("Not found", 0, inplace=True)
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    
    return df



In [ ]:
header_out = '''{
    "row_header" : ["Home Team","Away Team"],
    "columns_headers" : ["Goals","Shots", "Fouls","Yellow Cards","Red Cards","Corner Kicks","Free Kicks","Offsides"]
}
'''
for idx in range(0,5):
    superset = paragraph_to_numbered_sentences(paragraph= df['text'][idx])
    sz = len(superset)
    set1 = superset[:int(sz/2)]
    set2 = superset[int(sz/2):]
    formatted_output_1 = "\n".join(set1)
    formatted_output_2 = "\n".join(set2)
    ## Call 1
    input_text = header_out + '\n' + formatted_output_1
    output_table = ask_chatgpt(text=input_text,prompt_path="prompts/Livesum/fill_table.txt")
    with open(f"model_outputs/Livesum/GPT4o_TableFill_2step/step1/{idx}.txt",'w+') as f:
       f.write(output_table)
    dfa = extract_half_table(output_table)
    ## Prep table as str for call 2
    table_input = [dfa.columns.tolist()] + dfa.values.tolist()
    input_table = table_input
    columns = input_table[0]  # Extract column names from the first row
    data = input_table[1:]    # Remove the header row from the data
    dfa = pd.DataFrame(data,columns=columns)
    input_str_pipe = "\n".join(["|".join(map(str, row)) for row in input_table])
    ## Call 2
    input_text = "Given Table: \n" + input_str_pipe + 'Statements: \n' + formatted_output_2
    final_output = ask_chatgpt(text=input_text,prompt_path="prompts/Livesum/fill_table_2.txt")
    with open(f"model_outputs/Livesum/GPT4o_TableFill_2step/step2/{idx}.txt",'w+') as f:
        f.write(final_output)
    print('Saved results for idx',idx)
    print("*****************************")

### Schema + Table Baseline

In [ ]:
for idx in range(0,5):
    formatted_output = "\n".join(paragraph_to_numbered_sentences(paragraph= df['text'][idx]))
    input_text = formatted_output 
    output_table = ask_chatgpt(text=input_text,prompt_path="prompts/Livesum/tabgen_baseline.txt")
    with open(f"model_outputs/Livesum/GPT4o_TabGenBaseline/{idx}.txt",'w+') as f:
        f.write(output_table)
    #print(output_table)
    print('Saved results for idx',idx)
    print("*****************************")
        
#print(output_table)

### TabGen: Header+Table Generation (1-step)

In [ ]:
for idx in range(0,5):
    formatted_output = "\n".join(paragraph_to_numbered_sentences(paragraph= df['text'][idx]))
    with open(f'model_outputs/Livesum/GPT4o_Headers/{idx}.txt','r') as f:
        header_out = f.read()
    input_text = header_out + '\n' + formatted_output 
    output_table = ask_chatgpt(text=input_text,prompt_path="prompts/Livesum/fill_table.txt")
    with open(f"model_outputs/Livesum/GPT4o_TabGen/{idx}.txt",'w+') as f:
        f.write(output_table)
    print('Saved results for idx',idx)
    print("*****************************")
        
#print(output_table)

### TabGen: Header+Table Generation (2-step)

In [23]:
for idx in range(4,5):
    superset = paragraph_to_numbered_sentences(paragraph= df['text'][idx])
    sz = len(superset)
    set1 = superset[:int(sz/2)]
    set2 = superset[int(sz/2):]
    formatted_output_1 = "\n".join(set1)
    formatted_output_2 = "\n".join(set2)
    ## Call 1
    with open(f"model_outputs/Livesum/GPT4o_Headers/{idx}.txt","r") as f:
        header_out = f.read()
    input_text = "Table Schema: \n" + header_out + '\n' + "Statements: \n" + formatted_output_1
    output_table = ask_chatgpt(text=input_text,prompt_path="prompts/Livesum/fill_table.txt")
    with open(f"model_outputs/Livesum/GPT4o_TabGen_2step/step1/{idx}.txt",'w+') as f:
       f.write(output_table)
    dfa = extract_half_table(output_table)
    print(dfa)
    ## Prep table as str for call 2
    table_input = [dfa.columns.tolist()] + dfa.values.tolist()
    input_table = table_input
    # columns = input_table[0]  # Extract column names from the first row
    # data = input_table[1:]    # Remove the header row from the data
    # dfa = pd.DataFrame(data,columns=columns)
    input_str_pipe = "\n".join(["|".join(map(str, row)) for row in table_input])
    ## Call 2
    input_text = "Given Table: \n" + input_str_pipe + 'Statements: \n' + formatted_output_2 
    final_output = ask_chatgpt(text=input_text,prompt_path="prompts/Livesum/fill_table_2.txt")
    with open(f"model_outputs/Livesum/GPT4o_TabGen_2step/step2/{idx}.txt",'w+') as f:
        f.write(final_output)
    print('Saved results for idx',idx)
    print("*****************************")

NameError: name 'extract_half_table' is not defined

In [30]:
#output_table = ask_chatgpt(text=input_text,prompt_path="prompts/Livesum/fill_table.txt")
idx1 = 0
with open(f"model_outputs/Livesum/GPT4o_TableFill_2step/step1/{idx1}.txt",'r') as f:
    output_2 = f.read()
dx = extract_half_table(output_2)

Row length mismatch! Header columns: 8, Row data: 9
Row content: ['Home Team', 'Not found', 'Not found', '7', '1', '5', '4', '2', '1']
Row length mismatch! Header columns: 8, Row data: 9
Row content: ['Away Team', 'Not found', 'Not found', '4', 'Not found', '7', '7', '3', '2']


ValueError: Row 1 has 8 values, but 7 were expected. Row content: [0, 0, '7', '1', '5', '4', '2', '1']